# Synthetic camera RAW 50 MP Colab

Incremental implementation of PRD_v3. Output will be a **synthetic sensor reconstruction of a hypothetical
exposure**, never a genuine camera capture. Runtime > Run all executes the whole chain into one `run_YYYYMMDD_HHMMSS` folder, validates it
(`validation/validation_report.html`), runs the 2K/4K comparison, builds a bundle zip and ends by delivering
the final image (the 16-bit TIFF and the DNG).
Metadata must carry D/I/F/M provenance; DNG Make must be `Synthetic`. No Sony MakerNotes,
realistic Sony serials, date inference or GPS inference are permitted.

Run in Colab only. Configure the panel before running Input audit. `FAST_DEV_RUN` reduces planned
sensor dimensions and shrinks the SR input by the same factor; SR reconstructs on disk using the planned sensor
geometry.
Learned weights need a Colab runtime; licence-gated ones also need explicit acceptance. Compute stages benchmark initial tiles,
checkpoint to host RAM/disk and release GPU tensors between stages.

## Environment + reproducibility

In [ ]:
# Colab setup. REPO_REF defaults to main, which carries every merged change. For a reproducible release,
# pin REPO_REF to the reviewed 40-character commit SHA of a commit on main (a pull request's head commit SHA
# also works for trying it before merge). A pinned SHA installs exactly or the cell fails; INSTALLED_REF
# records what was installed.
import importlib
import re
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/RDC28/experiment_50MP.git"
REPO_REF = "main"
if REPO_REF != "main" and not re.fullmatch(r"[0-9a-fA-F]{40}", REPO_REF):
    raise ValueError(f"REPO_REF must be 'main' or a full 40-character commit SHA, got {REPO_REF!r}")

# The sr extra adds spandrel (and torch, which Colab preinstalls) so SwinIR and Real-ESRGAN can run.
PACKAGE = "synthetic-raw-50mp-lab[sr]"
if subprocess.call([sys.executable, "-m", "pip", "install", f"{PACKAGE} @ git+{REPO_URL}@{REPO_REF}"]) != 0:
    raise RuntimeError(f"Could not install {PACKAGE} at {REPO_REF!r}; check REPO_REF and the network")
INSTALLED_REF = REPO_REF
print(f"Installed {PACKAGE} from {REPO_URL}@{INSTALLED_REF}")
# This repository distribution installs the top-level synthraw package.


def on_colab():
    try:
        import google.colab  # noqa: F401
    except ImportError:
        return False
    return True


# On Colab, the pinned current ExifTool release (EXIFTOOL_VERSION in synthraw.sony_registry, downloaded from
# the exiftool.org release link, sha256-verified before use and run with the system Perl) builds
# metadata/sony_tag_registry.json and records its version. If the download or the verification fails, ExifTool
# counts as absent: the registry uses the bundled seed (seed_only: true) and records the reason.
if on_colab():
    importlib.invalidate_caches()
    # Learned SR needs both; a missing one fails here rather than when the SR stage starts.
    for module in ("torch", "spandrel"):
        importlib.import_module(module)
    from synthraw.sony_registry import install_exiftool

    install_exiftool("/content/exiftool")
print(f"ExifTool: {shutil.which('exiftool') or 'not installed (Sony tag registry falls back to the bundled seed)'}")

### Configuration

Every field comes from `ExperimentConfig`. Edit controls, then run Input audit to validate and freeze
settings. PATH uses a runtime path. DRIVE mounts Drive and uses INPUT_PATH inside that mount.
CALIBRATION_ASSETS, DARK_FRAME_PATHS and FLAT_FIELD_PATHS accept JSON lists of runtime paths. FULL_CALIBRATION requires REFERENCE_RAW_PATH, matched dark pairs, and consecutive flat pairs at two or more distinct unsaturated levels. Capture darks and uniformly illuminated flats at matching exposure, temperature and gain. Leave INPUT_PATH empty for UPLOAD.
If ipywidgets is unavailable, edit `CONFIG_VALUES` in this cell and rerun it. Changing configuration
requires rerunning Input audit and subsequent cells to start a fresh run.

In [ ]:
import json
from dataclasses import fields
from typing import Literal, get_args, get_origin, get_type_hints

from IPython.display import display

from synthraw.config import ExperimentConfig

CONFIG_VALUES = ExperimentConfig().to_dict()
controls = {}
try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

if widgets is None:
    print("ipywidgets unavailable: edit CONFIG_VALUES above, then run Input audit.")
    print(json.dumps(CONFIG_VALUES, indent=2))
else:
    hints = get_type_hints(ExperimentConfig)
    for field in fields(ExperimentConfig):
        name, value, hint = field.name, CONFIG_VALUES[field.name], hints[field.name]
        options = dict(description=name, style={"description_width": "260px"}, layout=widgets.Layout(width="100%"))
        if get_origin(hint) is Literal:
            control = widgets.Dropdown(options=get_args(hint), value=value, **options)
        elif hint is bool:
            control = widgets.Checkbox(value=value, **options)
        elif hint is int:
            # Text avoids imposing widget integer limits on NumPy seed values.
            control = widgets.Text(value=str(value), **options)
        elif hint is float:
            control = widgets.FloatText(value=value, **options)
        else:
            path_list = name in ("CALIBRATION_ASSETS", "DARK_FRAME_PATHS", "FLAT_FIELD_PATHS")
            text = json.dumps(value) if path_list else (value or "")
            control = widgets.Text(value=text, **options)
        controls[name] = control
    display(widgets.VBox(list(controls.values())))


def read_configuration():
    """Validate current controls, including dependent choices, before any input I/O."""
    if not controls:
        return ExperimentConfig.from_dict(CONFIG_VALUES)
    values = {name: control.value for name, control in controls.items()}
    for field in fields(ExperimentConfig):
        if get_type_hints(ExperimentConfig)[field.name] is int:
            values[field.name] = int(values[field.name])
    values["INPUT_PATH"] = values["INPUT_PATH"] or None
    values["REFERENCE_RAW_PATH"] = values["REFERENCE_RAW_PATH"] or None
    for name in ("CALIBRATION_ASSETS", "DARK_FRAME_PATHS", "FLAT_FIELD_PATHS"):
        values[name] = json.loads(values[name])
    return ExperimentConfig.from_dict(values)

In [ ]:
from synthraw.environment import EnvironmentInspector

environment = EnvironmentInspector().inspect()
EnvironmentInspector.print_report(environment)

### Mandatory tiny RAW gate

Run this before Input audit or any reconstruction. Sixteen 64 x 48 Bayer fixtures test all CFA
phases at both ADC depths, including the experimental virtual 16-bit profile, in both DNG layouts
(uncompressed and lossless JPEG). Each DNG must reopen with identical pixels and matching metadata
through independent LibRaw and develop to finite 16-bit RGB. Any failure stops execution; fix the runtime before continuing. Temporary
DNGs are removed on success or failure. These fixtures are synthetic test data, not the input image.

In [ ]:
# Mandatory RAW gate: only 64 x 48 fixtures, before any reconstruction.
from dataclasses import replace
from tempfile import TemporaryDirectory

import numpy as np

from synthraw.dng_metadata import CFA_PATTERNS, DNGMetadataBuilder
from synthraw.dng_writer import DNGWriter
from synthraw.libraw_validator import LibRawValidator

RAW_GATE_PASSED = False


def run_raw_gate():
    """Fail immediately if independent LibRaw decoding or development fails."""
    from pathlib import Path

    from synthraw.geometry import TargetGeometryPlanner

    tiny_geometry = replace(
        TargetGeometryPlanner().plan(64, 48),
        width=64,
        height=48,
        active_area=(0, 0, 48, 64),
        default_crop_origin=(0, 0),
        default_crop_size=(64, 48),
    )
    with TemporaryDirectory(prefix="synthraw_gate_") as directory:
        for bits, compression in ((14, "UNCOMPRESSED"), (16, "UNCOMPRESSED"), (14, "LOSSLESS"), (16, "LOSSLESS")):
            gate_config = ExperimentConfig(ADC_BITS=bits, DNG_COMPRESSION=compression)
            for phase in CFA_PATTERNS:
                records = DNGMetadataBuilder().build(
                    tiny_geometry,
                    gate_config,
                    cfa_phase=phase,
                    as_shot_neutral=(0.5, 1, 0.75),
                    color_matrix2=((0.8, 0.1, 0.05), (0.05, 1, 0.1), (0.02, 0.1, 0.9)),
                )
                mosaic = np.arange(48 * 64, dtype=np.uint16).reshape(48, 64) + 512
                path = Path(directory) / f"{phase}_{bits}_{compression}.dng"
                DNGWriter().write(path, mosaic, records, gate_config)
                LibRawValidator().validate(path, mosaic, records, develop=True)


run_raw_gate()
RAW_GATE_PASSED = True
print(
    "RAW gate passed: all four Bayer phases at 14 and virtual 16 bits, uncompressed and lossless JPEG; "
    "LibRaw pixels, metadata and RGB verified."
)

## Input audit

In [ ]:
if not globals().get("RAW_GATE_PASSED", False):
    raise RuntimeError("Run the mandatory tiny RAW gate before Input audit")

from pathlib import Path

from synthraw.pipeline import PipelineRun

config = read_configuration()
if config.INPUT_MODE == "UPLOAD":
    from google.colab import files

    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Upload exactly one JPEG or PNG")
    source_path = Path(next(iter(uploaded)))
    del uploaded  # Colab already saved the bytes to source_path.
else:
    if config.INPUT_MODE == "DRIVE":
        from google.colab import drive

        drive.mount(config.DRIVE_MOUNT_PATH)
    source_path = Path(config.INPUT_PATH).expanduser()
    if config.INPUT_MODE == "DRIVE" and not source_path.resolve().is_relative_to(
        Path(config.DRIVE_MOUNT_PATH).resolve()
    ):
        raise ValueError("DRIVE INPUT_PATH must be inside DRIVE_MOUNT_PATH")

if globals().get("pipeline") is not None:
    globals()["pipeline"].close()  # Rerunning Input audit starts a fresh run folder.
# Every section below runs one stage of this PipelineRun, the same stage code as run_pipeline.
pipeline = PipelineRun.create(source_path, config, "/content/synthraw_runs", environment=environment)
run, execution_log = pipeline.run, pipeline.logger
pipeline.run_stage("input_audit")
audit = pipeline.audit
execution_log.info(f"Input audit: {source_path.name} -> {pipeline.source_original.name}; sha256 {audit.sha256}")
print(json.dumps(audit.to_dict(), indent=2))
print(f"Run directory: {run.root}")

## Profile selection

In [ ]:
print(f"Target profile: {config.TARGET_PROFILE}")
print(f"Profile source: {config.CAMERA_PROFILE_SOURCE}; spectral mode: {config.SPECTRAL_MODE}")
# The camera section below validates the selected profile and writes its provenance.

## Target geometry

In [ ]:
from dataclasses import asdict

pipeline.run_stage("geometry")
print(json.dumps(asdict(pipeline.geometry_plan), indent=2))
print("Geometry and memory estimates only; no full sensor arrays allocated.")

## Source diagnostics

In [ ]:
print(f"Source: {audit.resolution_class}, aspect {audit.aspect_ratio:.6f}")
print(f"Near-black RGB fractions: {audit.near_black_fractions}")
print(f"Near-white RGB fractions: {audit.near_white_fractions}")
# Reconstruction milestone: add spatial detail, ringing and compression diagnostics.

## SR reconstruction

In [ ]:
# Pinned permissive checkpoints (SwinIR, Real-ESRGAN) download inside Colab without this flag.
# Licence-gated backends and experimental stubs stay disabled unless you review their licence and set True.
# If the chosen SWINIR or REAL_ESRGAN cannot load (a missing package, no GPU, a failed download), this cell
# stops with the reason instead of silently using another backend. SR_BACKEND = "LANCZOS" needs no weights.
ACCEPT_SR_MODEL_LICENCE = False

pipeline.options["accept_licence"] = ACCEPT_SR_MODEL_LICENCE
pipeline.run_stage("super_resolution")
sr_result = pipeline.sr_result
print(f"SR backend: {sr_result.tiled.backend} at x{sr_result.scale_plan.scale}; fallbacks: {pipeline.fallbacks}")
print(f"SR input: {sr_result.sr_input_size[0]} x {sr_result.sr_input_size[1]} (FAST_DEV_RUN: {config.FAST_DEV_RUN})")
print(
    f"Tile benchmark: {sr_result.tiled.benchmark_tile_count} tiles; estimated tile seconds: "
    f"{sr_result.tiled.estimated_seconds:.2f}"
)
print(f"SR canvas: {sr_result.canvas.path}; source fidelity: {sr_result.source_fidelity}")

## HDR / scene-linear reconstruction

In [ ]:
import matplotlib.pyplot as plt

pipeline.run_stage("hdr")
hdr_result = pipeline.hdr_report

# Preview only a bounded sample; never materialize a full sensor-sized plotting array.
sr_rgb = np.load(sr_result.canvas.path, mmap_mode="r")
preview_step = max(1, int(np.ceil(max(sr_rgb.shape[:2]) / 256)))
preview = np.asarray(sr_rgb[::preview_step, ::preview_step], dtype=np.float32)
if sr_rgb.dtype == np.uint8:
    preview /= 255.0
fig, ax = plt.subplots(figsize=(6, 3))
for channel, colour in enumerate(("red", "green", "blue")):
    ax.hist(preview[..., channel].ravel(), bins=64, range=(0, 1), histtype="step", color=colour)
ax.set(title="SR encoded RGB histogram (sampled)", xlabel="Encoded value", ylabel="Sample count")
fig.tight_layout()
fig.savefig(run.directory("validation") / "input_histograms.png", dpi=100)
plt.close(fig)
fig, axes = plt.subplots(1, 2, figsize=(7, 3))
for ax, name in zip(axes, ("highlight_fabricated_mask", "shadow_fabricated_mask"), strict=True):
    mask = np.load(run.directory("confidence") / f"{name}.npy", mmap_mode="r")
    ax.imshow(mask[::preview_step, ::preview_step], vmin=0, vmax=1, cmap="gray")
    ax.set_title(name.replace("_fabricated_mask", " uncertainty"))
    ax.axis("off")
fig.tight_layout()
fig.savefig(run.directory("validation") / "clipping_masks.png", dpi=100)
plt.close(fig)
fig, ax = plt.subplots(figsize=(5, 3))
if sr_result.seams is not None:
    seam_map = np.load(sr_result.seams.residual_map_path, mmap_mode="r")
    seam_step = max(1, int(np.ceil(max(seam_map.shape) / 256)))
    plot = ax.imshow(seam_map[::seam_step, ::seam_step], cmap="magma", vmin=0)
    fig.colorbar(plot, ax=ax, label="Mean absolute RGB residual")
else:
    ax.text(0.5, 0.5, sr_result.seam_unavailable_reason, ha="center", va="center", wrap=True)
ax.set_title("SR tile seam diagnostic (sampled)")
ax.axis("off")
fig.tight_layout()
fig.savefig(run.directory("validation") / "tile_seam_map.png", dpi=100)
plt.close(fig)
print(f"Generated HDR area: {hdr_result['generated_hdr_area_percentage']:.2f}% (inferred/fabricated radiance)")
print(f"Gamut mapping audit: {hdr_result['gamut_mapping']}")

## Camera geometry + exposure inference

In [ ]:
# Explicit hypothetical settings, never recovered capture facts. REFERENCE_RAW and FULL_CALIBRATION
# read the reference file from the REFERENCE_RAW_PATH configuration field.
MANUAL_WB = None  # Or {"manual_cct": 5500.0, "manual_tint": 0.0}.
MANUAL_EXPOSURE = None  # Required for MANUAL: (seconds, f-number, ISO).
SCENE_EV100 = 10.0  # Scene-scale prior, not brightness metering.
MANUAL_FOV_DEGREES = None

pipeline.options.update(
    manual_wb=MANUAL_WB,
    manual_exposure=MANUAL_EXPOSURE,
    scene_ev100=SCENE_EV100,
    manual_fov_degrees=MANUAL_FOV_DEGREES,
)
pipeline.run_stage("camera_exposure")
print("Hypothetical exposure candidates (relative heuristic weights):")
print("Seconds       F-number   ISO       Weight")
for candidate in pipeline.exposure.candidates:
    print(f"{candidate.exposure_time:<13.6g} {candidate.f_number:<10g} {candidate.iso:<9d} {candidate.weight:.6f}")

## Inverse ISP / camera colour

In [ ]:
# HDR already undid tone processing; the inverse ISP keeps its radiance and headroom.
# EXPERIMENTAL / LOW CONFIDENCE: spectral inference is a licence-gated stub that keeps MATRIX_ONLY.
pipeline.run_stage("inverse_isp")
print(f"Camera-native RGB saved; backend: {pipeline.inverse_isp_backend}")

## Optical forward model

In [ ]:
pipeline.run_stage("optics")
print(f"Optical sensor RGB saved; optical simulation enabled: {config.OPTICAL_SIMULATION}")

## Sensor/electron simulation

In [ ]:
# The leading ISO hypothesis sets the analogue gain, and the exposure scale keeps the HDR
# highlight headroom (2^HDR_HIGHLIGHT_EV) below full well at that gain.
pipeline.run_stage("electrons_noise")
sensor_parameters = pipeline.camera_profile.sensor
print(
    f"Expected and noisy electrons saved: ISO {pipeline.exposure.records['PhotographicSensitivity'].value}, "
    f"analogue gain {sensor_parameters['applied_analogue_gain'].value:g}, "
    f"exposure scale {sensor_parameters['applied_exposure_scale'].value:.6g}."
)

## CFA construction

In [ ]:
pipeline.run_stage("cfa")
print(f"Clean and noisy {pipeline.cfa_phase} CFA intermediates saved in electrons.")

## ADC quantisation

In [ ]:
pipeline.run_stage("adc")
print(f"ADC codes and RAW uint16 mosaic saved; applied NoiseProfile: {pipeline.adc_result.noise_profile}")

## Metadata construction

In [ ]:
# Built from the applied ADC, colour and exposure records, so reruns cannot reuse stale noise metadata.
pipeline.run_stage("metadata")
print("DNG provenance, metadata.json and Sony sidecars written; the DNG still requires independent validation.")

## DNG export

In [ ]:
# Staged on the same filesystem; only a LibRaw-validated DNG is published below.
pipeline.run_stage("dng_export")
print("DNG written to staging; the independent LibRaw round trip publishes it.")

## RAW round-trip development

In [ ]:
# Independent LibRaw reopen plus default AHD development (PRD validation steps 1-2). A failure is a
# hard failure: the DNG is not published. Demosaic variants and editability run in the Validation suite.
pipeline.run_stage("libraw_reopen")
dng_path = pipeline.dng_path
print(
    f"LibRaw reopened {dng_path.name} ({config.DNG_COMPRESSION}) with identical pixels and developed it to "
    "finite 16-bit RGB."
)

# N_SENSOR_REALISATIONS > 1 publishes further noise realisations of the same exposure and sensor body, each
# with its own recorded SENSOR_NOISE_SEED and its own LibRaw gate (validation/sensor_realisations.json).
pipeline.run_stage("sensor_realisations")
for entry in pipeline.realisations:
    print(f"Realisation {entry['index']}: SENSOR_NOISE_SEED {entry['sensor_noise_seed']} -> {entry['dng']}")

## 16-bit TIFF export

In [ ]:
# The TIFF is developed by LibRaw from the published DNG, never from any pre-sensor RGB stage.
pipeline.run_stage("raw_development_tiff")
pipeline.run_stage("exr_preview")
for path in (pipeline.tiff_path, pipeline.exr_path, pipeline.preview_path):
    if path is not None:
        print(f"Master written: {path}")

## Validation suite

Writes `metadata/provenance.json` (Metadata construction already wrote `metadata/metadata.json`), then runs
the PRD automated validation sequence on the finished run tree: numeric health, CFA phase, ADC contract,
LibRaw reopen, metadata dependencies, colour matrices, the flat-field mean-variance noise test against the
DNG NoiseProfile, demosaic variants, -4..+4 EV and cool/neutral/warm WB editability, round-trip
SSIM/deltaE00, seams, HDR radiance and seed checks. Results go to `validation/validation_results.json`, nine
figures and a self-contained `validation/validation_report.html`. A failed hard gate (including NoiseProfile
agreement, demosaic variants and the exposure and WB stress tests) yields a FAILED report and a FAILED
`run_report.json`; QA targets are listed with measured values but never tuned to pass.

In [ ]:
from synthraw.report import validate_and_report

C2PA_SIGNER = None  # A configured c2pa-python signer; without one C2PA is recorded as unavailable.

pipeline.options["c2pa_signer"] = C2PA_SIGNER
# Validation and the 2K/4K comparison read only the finished run tree, so provenance.json (with
# optional C2PA on the masters) is frozen before any report is written.
pipeline.run_stage("provenance")
validation_run = validation_report_path = None
if not config.WRITE_VALIDATION_REPORT:
    print("Validation disabled by WRITE_VALIDATION_REPORT; the run report records it as not run.")
elif not config.WRITE_METADATA_JSON:
    print("Validation reads metadata/metadata.json; enable WRITE_METADATA_JSON to validate this run.")
else:
    # Numeric health, CFA, ADC, LibRaw, metadata, colour, flat-field noise, demosaic variants,
    # -4..+4 EV and WB editability, round trip, seams, HDR and seed checks, then figures and HTML.
    validation_run, validation_report_path = validate_and_report(run.root, log=execution_log.info)
    for check in validation_run.checks:
        result = {True: "pass", False: "FAIL", None: "report"}[check.passed]
        print(f"{'HARD' if check.hard else 'QA  '} {result:<6} {check.domain:<12} {check.name}")
    missed = validation_run.to_dict()["qa_targets_missed"]
    print(f"Validation {validation_run.status}; QA targets missed: {missed or 'none'}")
    print(f"Report: {validation_report_path}")

## 2K/4K comparison

Runs a second source through the identical chain (same config, seeds, hypotheses and FAST_DEV size)
so only source resolution varies, then writes `validation/comparison_2k_vs_4k.html` and `.json`.
Set `COMPARISON_SECOND_INPUT` to the other-resolution export of the same composition; otherwise a
half-resolution Lanczos copy of this input is derived and labelled as derived in the report. The second
run is written beside this run (`<run>_comparison/`) and is not bundled. This doubles Colab compute.

In [ ]:
from synthraw.comparison import compare_and_report, make_downsampled_source
from synthraw.pipeline import run_pipeline

# PRD 2K-versus-4K experiment: a second source runs through the identical chain and settings;
# only source resolution may vary. It doubles Colab compute, so set False to skip it.
RUN_SOURCE_RESOLUTION_COMPARISON = True
# Runtime path to the same composition at another resolution, e.g. the 4K export of a 2K input.
# None derives a half-resolution Lanczos copy of this input, labelled as derived in the report.
COMPARISON_SECOND_INPUT = None
# Optional named crops as (x0, y0, x1, y1) frame fractions, e.g. {"face_skin": (0.4, 0.2, 0.6, 0.5)};
# None uses a 2 x 3 grid. Suggested PRD names: face_skin, hair, saree_fabric, jewellery, smoke, bokeh_light.
COMPARISON_CROPS = None

comparison = comparison_report_path = None
if not RUN_SOURCE_RESOLUTION_COMPARISON:
    print("2K/4K comparison skipped by RUN_SOURCE_RESOLUTION_COMPARISON.")
elif not config.WRITE_TIFF16 or not config.WRITE_METADATA_JSON:
    print("2K/4K comparison needs the 16-bit TIFF master and metadata.json.")
else:
    comparison_root = run.root.parent / f"{run.root.name}_comparison"
    comparison_root.mkdir(exist_ok=True)
    low_source_derived = COMPARISON_SECOND_INPUT is None
    if low_source_derived:
        second_source = make_downsampled_source(
            pipeline.source_original, comparison_root / "derived_half_resolution.png"
        )
        print(f"Single input: derived {second_source.name} from this source (labelled derived in the report).")
    else:
        second_source = Path(COMPARISON_SECOND_INPUT).expanduser()
    # This run's own options (hypotheses, FAST_DEV size, strips, licence) lock every setting.
    second_run = run_pipeline(second_source, config, comparison_root, environment=environment, **pipeline.options)
    execution_log.info(f"Comparison run written: {second_run.root}")
    comparison, comparison_report_path = compare_and_report(
        run.root,
        second_run.root,
        run.directory("validation"),
        crops=COMPARISON_CROPS,
        low_source_derived=low_source_derived,
        log=execution_log.info,
    )
    print(f"2K/4K comparison {comparison.status}; flags: {comparison.flags or 'none'}")
    if comparison.locked_mismatches:
        print(f"Locked settings that differ: {sorted(comparison.locked_mismatches)}")
    print(f"Report: {comparison_report_path}")

## Provenance + report packaging

In [ ]:
from synthraw.bundle import bundle_run, copy_run_to_drive, deliver_bundle, drive_run_folder, sync_to_drive

# The browser zip leaves out the full-resolution intermediates/ and confidence/ arrays (listed in
# logs/bundle_manifest.json); set False to include them. The whole run folder, intermediates included, is
# copied to Drive: when they are left out of the zip, Drive is mounted here if it is not mounted already.
BUNDLE_EXCLUDE_LARGE_ARRAYS = True
BROWSER_DOWNLOAD_MAX_MB = 2048  # Larger zips are not offered as a browser download.

drive_dir, drive_warning = drive_run_folder(config.DRIVE_MOUNT_PATH, mount=BUNDLE_EXCLUDE_LARGE_ARRAYS)
report_warnings = [drive_warning] if drive_warning else []
if drive_dir is not None:
    _, copy_warning = copy_run_to_drive(run.root, drive_dir)
    report_warnings += [copy_warning] if copy_warning else []
# A failed validation hard gate never yields a success report; the bundle still carries the evidence.
run_report = pipeline.write_report(validation=validation_run, comparison=comparison, warnings=report_warnings)
bundle_path = bundle_run(run.root, exclude_large_arrays=BUNDLE_EXCLUDE_LARGE_ARRAYS)
execution_log.info(f"Bundle written: {bundle_path.name}")
sync_warning = None if drive_dir is None else sync_to_drive(run.root, drive_dir, bundle_path)
delivery = deliver_bundle(bundle_path, max_download_bytes=BROWSER_DOWNLOAD_MAX_MB * 1024**2)
for warning in report_warnings + ([sync_warning] if sync_warning else []):
    print(f"WARNING: {warning}")
print(f"C2PA: {pipeline.c2pa['status']}; validation: {run_report['validation']}")
if run_report["status"] == "FAILED":
    print(f"VALIDATION FAILED: hard gates {run_report['hard_failures']}; see validation/validation_report.html")
print(f"Bundle: {bundle_path} ({bundle_path.stat().st_size / 1024**2:.1f} MB); delivery: {delivery}")

## Final image

The run's final image is `masters/developed_50mp_16bit.tif` (LibRaw development of the published DNG, Adobe RGB)
together with `raw/synthetic_50mp.dng`. This cell shows the preview, prints both paths, offers both files as
browser downloads and, when the bundle cell copied the run to Google Drive, prints their paths in the Drive run
folder (`MyDrive/synthraw_runs/<run>/`).

In [ ]:
from IPython.display import Image as PreviewImage

from synthraw.bundle import deliver_final_image

# The final image is the LibRaw-developed 16-bit TIFF master plus the synthetic DNG it was developed from.
# Each is offered as a browser download (up to BROWSER_DOWNLOAD_MAX_MB; a full 50 MP TIFF is about 300 MB, so
# it takes a few minutes); its Drive path is printed when the bundle cell copied the run to Drive. Set False to
# skip the browser downloads; the printed local paths stay valid until this Colab runtime recycles.
DOWNLOAD_FINAL_IMAGE = True

if pipeline.preview_path is not None and pipeline.preview_path.is_file():
    display(PreviewImage(filename=str(pipeline.preview_path), width=960))
final_image = deliver_final_image(
    run.root,
    drive_dir=drive_dir,
    download=DOWNLOAD_FINAL_IMAGE,
    max_download_bytes=BROWSER_DOWNLOAD_MAX_MB * 1024**2,
)